1. Shape of embedding [n_t]
2. Shape of output[n_vocab]

Build the network: x_n -> f(x_n) -> error -> x_n-1 -> 
For each example, run the inference pass, then run the backward learning pass 

For inference: 
1. Get the input
2. Initialise the layers to a random value 
3. Pad the weights & the layers [do this beforehand]
4. Calculate the errors, pre-activation for all layers
5. Get the gradient
6. Update layers using gradient

For training: 
1. Get the errors & pre-activations for all layers 
2. Get the gradient
3. Update the weights using the gradient 

For per minibatch update training: 
1. Initialise the weights randomly
2. Get the training samples & fix input layers to that
3. Fix the input layer to that 
4. Add necessary padding
5. Run the traing cycle per sample
6. Batch the training update process across multiple samples using vmap

For each epochs - run for n epochs: 
1. Get the list of minibatches from the training batch
2. Run the training for each minibatch
3. Updates weights 

TODO (in order): 
1. Replicate results from the CFAIR training described in https://arxiv.org/pdf/2506.06332
2. Training details - how to interleave inference & weight updates, accelarating training via parallel kernels, batching over samples
3. Training over the minibatch - gradient is on the basis of avg. grad across the batch (but is avg the only way)
4. Can PC learning be applied to transformers -> learning attention is the issue, "attention-like patterns", where the entire text is processed via linear layers
5. Building an RNN network trained for auto-regressive text generation using PCN

In [15]:
import jax.numpy as jnp
import jax
import pickle

def unpickle(file):
    with open(file, 'rb') as fo:
        dict = pickle.load(fo, encoding='bytes')
    return dict

# layers shape: [nl+1,nd_i]
# The nl layer is the output reference
# 0 layer is the input signal
# weights shape: [n_l,nd_i-1,nd_i]
# the n_l weight are the output weights, 
# not used for this calc
## layers proceed from y, x_l.... x_0
## layers proceed from W_o, W_l-1 .... W_0
## Errors are from e_o, e_l-1....e_0
def errorsnpreactivations(layers: jnp.array, weights: jnp.array):
    first_layer_index = layers.shape[0] - 1
    ## Just getting the upper n layers
    upper_layers = layers[1:first_layer_index,:]
    upper_weights = weights[1:]
    # upper layers shape: [n_l-1,nd]
    # upper_weights shape: [n_l-1,nd,nd]
    # preactivation shape: [n_l-1,nd]
    ##TO FIX: Is the matrix multiplication as expected?
    print("upper_weights & upper_layers:",upper_weights,upper_layers)
    preactivations = jnp.einsum("bij,bj->bi", upper_weights, upper_layers)
    activation = jax.nn.relu(preactivations)
    lower_layers = layers[2:first_layer_index+1,:]
    # errors shape: [n_l-1,nd]
    errors = activation - lower_layers
    # print("Returning errorsnpreactivations", errors, preactivations)
    return errors, preactivations

def mse(errors: jnp.array):
    jnp.sum(jnp.square(errors))

## Consider the first layer to be the fixed output itself
def add_supervised_error(layers,weights,errors): 
    output = layers[0]
    output_layer = layers[1]
    output_weight = weights[0]
    last_error_index = len(errors)-1
    errors_shifted = errors[0:last_error_index,:]
    print("output & output_weight & output_layer:",output,output_weight,output_layer)
    output_error = jnp.expand_dims(output - output_weight @ output_layer,0)
    print("output_error & errors_shifted",output_error,errors_shifted)
    return jnp.append(output_error,errors_shifted,0)
    
##Layers shape: [nl+1,nd]; without padding [nl+1,nd_i]
## Errors shape: [nl-1,nd]; without padding [nl-1,nd_i-1]
## Preactivation shape: [nl-1,nd]; without padding [nl-1,nd_i-1]
## Weights shape: [nl, nd, nd]; without padding [n_l,nd_i-1,nd_i]
##TODO: Modify to consider the first layer & weight to be for outputs??
def update_layers(layers: jnp.array, weights: jnp.array, errors: jnp.array, preactivations: jnp.array, eta_inference):
    #Instead of padding shifted errors with 0, fill it with supervised errors now
    # errors_shifted = jnp.pad(errors[0:len(errors)-1,:],((0,1),(0,0)),mode='empty')
    errors_shifted = add_supervised_error(layers,weights,errors)
    # errors_shifted shape: [nl,nd]
    # jax.debug.print("Errors shifted in update_layers {}",errors_shifted)
    first_layer_index = layers.shape[0] - 1
    upper_layers = layers[1:first_layer_index,:]
    upper_weights = weights[1:]
    drelu = lambda x: (x > 0).astype(x.dtype)
    print("layers,weights,errors,preactivations: ",layers,weights,errors,preactivations)
    print("upper_weights & errors & preactivations:",upper_weights,errors,preactivations)
    ##TO FIX: Potential bug here, the vector may be transposed along the batch dimension, avoid that:
    ##TO FIX: Is the matrix multiplication as expected?
    grad = errors_shifted - jnp.einsum("bij,bj->bi", jnp.transpose(upper_weights,(0,2,1)),(errors * drelu(preactivations)))
    print("upper_layers & eta_inference & grad:",upper_layers,eta_inference,grad)
    upper_layers = upper_layers - eta_inference * grad
    # Once done return the layers: 
    output_layer = jnp.expand_dims(layers[0],0)
    input_layer = jnp.expand_dims(layers[first_layer_index],0)
    print("output_layer upper_layers input_layer",output_layer,upper_layers,input_layer)
    layers = jnp.append(output_layer,jnp.append(upper_layers,input_layer,0),0)
    jax.debug.print("Layers from update_layers {}",layers)
    return layers

def get_updated_grad(layers,weights,errors,preactivations):
    drelu = lambda x: (x > 0).astype(x.dtype)
    first_layer_index = layers.shape[0] - 1
    upper_layers = layers[1:first_layer_index,:]
    ##TO FIX: Potential bug here, the vector may be transposed along the batch dimension, avoid that:
    error_mod = -(errors * drelu(preactivations))
    upper_layers_t = upper_layers
    grad = jnp.einsum("bi,bj->bij",error_mod,upper_layers_t)
    print("grad, errors, layers, preactivations, error_mod,upper_layers_t",grad,errors,layers,preactivations,error_mod,upper_layers_t)
    sup_error = add_supervised_error(layers,weights,errors)[0]
    print("sup_error,layers[1]",sup_error,layers[1])
    grad_o = jnp.expand_dims(jnp.einsum("i,j->ij",sup_error,layers[1]),0)
    print("grad_o,grad",grad_o,grad)
    ## TO FIX: Check if the grad tensor takes the correct shape
    grads = jnp.append(grad_o,grad,0)
    return grads

def inference_updates(ts,layers,weights,eta_inference):
    def inference_update(params,t):
        layers = params[0]
        weights = params[1]
        eta_inference = params[2]
        errors, preactivation = errorsnpreactivations(layers,weights)
        layers = update_layers(layers,weights,errors,preactivation,eta_inference)
        return ((layers, weights,eta_inference),())
    return jax.lax.scan(inference_update,(layers,weights,eta_inference),ts)
 
def compute_grads(layers,weights):
    errors, preactivation = errorsnpreactivations(layers,weights)
    ## Note: no need for updating weights here? 
    grads = get_updated_grad(layers,weights,errors,preactivation)
    jax.debug.print("Grads from compute_grads {}",grads)
    return grads

def train_sample(ts,layers,weights,etas): 
    # jax.debug.print("Etas: {}",etas)
    print("Num inference: ",ts)
    end_state = inference_updates(ts,layers,weights,etas[0])
    # jax.debug.print("End state {}",end_state)
    layers = end_state[0][0]
    grads = compute_grads(layers,weights)
    return grads

def initialise_weights(key,num_input_size,num_output_size,num_hidden_size,pad_size): 
    input_weight_m = jnp.pad(jax.random.uniform(key,(1,num_hidden_size,num_input_size)),((0,0),(0,pad_size - num_hidden_size),(0,pad_size - num_input_size)),mode='empty')
    hidden_weight_m = jnp.pad(jax.random.uniform(key,(1,num_hidden_size,num_hidden_size)),((0,0),(0,pad_size - num_hidden_size),(0,pad_size - num_hidden_size)),mode='empty')
    output_weight_m = jnp.pad(jax.random.uniform(key,(1,num_output_size,num_hidden_size)),((0,0),(0,pad_size - num_output_size),(0,pad_size - num_hidden_size)),mode='empty')
    output_ref_weight_m = jnp.pad(jax.random.uniform(key,(1,num_output_size,num_output_size)),((0,0),(0,pad_size - num_output_size),(0,pad_size - num_output_size)),mode='empty')
    weights = jnp.append(jnp.append(output_weight_m,hidden_weight_m,0),input_weight_m,0)
    weights = jnp.append(output_ref_weight_m,weights,0)
    # print("Weights: ",weights.shape,weights)
    return weights

##Dataset should have (input,output) pairs
def initialise_batch_layers(dataset,key,num_input_size,num_output_size,num_hidden_layers,num_hidden_size,pad_size,num_batch_size): 
    dataset_input = dataset[0]
    dataset_output = dataset[1]
    # print("Dataset shapes",dataset_input.shape,dataset_output.shape)
    input_layer = jnp.expand_dims(dataset_input,1)
    hidden_layers = jnp.pad(jax.random.uniform(key,(num_batch_size,num_hidden_layers,num_hidden_size)),((0,0),(0,0),(0,pad_size - num_hidden_size)),mode='empty')
    output_layer = jnp.pad(jax.random.uniform(key,(num_batch_size,1,num_output_size)),((0,0),(0,0),(0,pad_size - num_output_size)),mode='empty')
    output_ref_layer = jnp.expand_dims(jnp.pad(dataset_output,((0,0),(0,pad_size - num_output_size)),mode='empty'),1)
    # print("Layer shapes",input_layer.shape,hidden_layers.shape,output_layer.shape,output_ref_layer.shape)
    layers = jnp.append(output_layer,hidden_layers,1)
    layers = jnp.append(layers,input_layer,1)
    layers = jnp.append(output_ref_layer,layers,1)
    # print("Layers: ",layers.shape,layers)
    return layers

def get_minibatch(batch,minibatch_size,i): 
    batch_size = batch.shape[0]
    index = int(i*(batch_size / minibatch_size))
    return batch[index:index+minibatch_size,:]
    
def run_training(dataset_input,dataset_output): 
    key = jax.random.key(434)
    num_epochs = 10
    ##Debug params: 
    # num_batch_size = 2
    # num_input_size = 6
    # num_output_size = 3
    # num_hidden_size = 5
    ##Train params: 
    num_input_size = 3072
    num_output_size = 10
    num_hidden_size = 100
    num_hidden_layers = 2
    pad_size = num_input_size
    # num_learn = 50
    num_learn = 1
    # ts_inference = jnp.array(range(10))
    ts_inference = jnp.array(range(1))
    etas = jnp.array([0.1,0.1])
    num_output_classes = 10
    # num_minibatch = 50
    num_minibatch = 1
    hotencoded_output = jax.nn.one_hot(dataset_output, num_output_classes)

    def run_epoch(weights): 
        for i in range(num_minibatch): 
            minibatch_input = get_minibatch(dataset_input,num_minibatch,i)
            minibatch_output = get_minibatch(hotencoded_output,num_minibatch,i)
            print("Running the num_minibatch iteration",i)
            for j in range(num_learn): 
                b_layers = initialise_batch_layers((minibatch_input,minibatch_output),key,num_input_size,num_output_size,num_hidden_layers,num_hidden_size,pad_size,num_minibatch)
                # print("b_layers shape:",b_layers)
                print("Running the num_learn iteration",j)
                b_grads = jax.jit(jax.vmap(train_sample,in_axes=(None,0,None,None),out_axes=0))(ts_inference,b_layers,weights,etas)
                weights = weights - etas[1] * jnp.mean(b_grads,0)

    weights = initialise_weights(key,num_input_size,num_output_size,num_hidden_size,pad_size)
    for i in range(num_epochs): 
        run_epoch(weights)

dataset = unpickle("data/cifar-10-batches-py/data_batch_1")
print(dataset[b"data"])
print(dataset[b"labels"])
dataset_input = jnp.array(dataset[b"data"][0:10,:])
dataset_output = jnp.array(dataset[b"labels"][0:10])
run_training(dataset_input,dataset_output)

/var/folders/34/pcpqkz1d73g417sphxj_z64c0000gn/T/ipykernel_90303/3607310568.py:7: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  dict = pickle.load(fo, encoding='bytes')


[[ 59  43  50 ... 140  84  72]
 [154 126 105 ... 139 142 144]
 [255 253 253 ...  83  83  84]
 ...
 [ 71  60  74 ...  68  69  68]
 [250 254 211 ... 215 255 254]
 [ 62  61  60 ... 130 130 131]]
[6, 9, 9, 4, 1, 1, 2, 7, 8, 3, 4, 7, 7, 2, 9, 9, 9, 3, 2, 6, 4, 3, 6, 6, 2, 6, 3, 5, 4, 0, 0, 9, 1, 3, 4, 0, 3, 7, 3, 3, 5, 2, 2, 7, 1, 1, 1, 2, 2, 0, 9, 5, 7, 9, 2, 2, 5, 2, 4, 3, 1, 1, 8, 2, 1, 1, 4, 9, 7, 8, 5, 9, 6, 7, 3, 1, 9, 0, 3, 1, 3, 5, 4, 5, 7, 7, 4, 7, 9, 4, 2, 3, 8, 0, 1, 6, 1, 1, 4, 1, 8, 3, 9, 6, 6, 1, 8, 5, 2, 9, 9, 8, 1, 7, 7, 0, 0, 6, 9, 1, 2, 2, 9, 2, 6, 6, 1, 9, 5, 0, 4, 7, 6, 7, 1, 8, 1, 1, 2, 8, 1, 3, 3, 6, 2, 4, 9, 9, 5, 4, 3, 6, 7, 4, 6, 8, 5, 5, 4, 3, 1, 8, 4, 7, 6, 0, 9, 5, 1, 3, 8, 2, 7, 5, 3, 4, 1, 5, 7, 0, 4, 7, 5, 5, 1, 0, 9, 6, 9, 0, 8, 7, 8, 8, 2, 5, 2, 3, 5, 0, 6, 1, 9, 3, 6, 9, 1, 3, 9, 6, 6, 7, 1, 0, 9, 5, 8, 5, 2, 9, 0, 8, 8, 0, 6, 9, 1, 1, 6, 3, 7, 6, 6, 0, 6, 6, 1, 7, 1, 5, 8, 3, 6, 6, 8, 6, 8, 4, 6, 6, 1, 3, 8, 3, 4, 1, 7, 1, 3, 8, 5, 1, 1, 4, 0, 9, 3, 7, 4, 